In [42]:
pip install joblib azureml.train.automl azureml.widgets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 129.3 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.9/987.9 kB 25.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 140.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 66.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 156.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 160.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.8/934.8 kB 36.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 84.3 MB/s  0:00:00
   ━━━━━━━━━━━━

In [14]:
from azureml.core import Workspace, Experiment

ws = Workspace.from_config()
exp = Experiment(workspace=ws, name="udacity-project")

print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep = '\n')

run = exp.start_logging()

Workspace name: quick-starts-ws-302116
Azure region: westeurope
Subscription id: d7f39349-a66b-446e-aba6-0053c2cf1c11
Resource group: aml-quickstarts-302116


In [15]:
from azureml.core.compute import ComputeTarget, AmlCompute

cluster_name = "Udacity-Project"

# TODO: Create compute cluster
# Use vm_size = "Standard_D2_V2" in your provisioning configuration.
# max_nodes should be no greater than 4.

### YOUR CODE HERE ###

compute_config = AmlCompute.provisioning_configuration(
    vm_size="Standard_D2_V2",
    max_nodes = 4
)

cpu_cluster = ComputeTarget.create(ws, cluster_name, compute_config)
cpu_cluster.wait_for_completion(show_output=True)

InProgress..
SucceededProvisioning operation finished, operation "Succeeded"
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned


In [16]:
from azureml.widgets import RunDetails
from azureml.train.sklearn import SKLearn
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.parameter_expressions import choice, uniform
from azureml.core import Environment, ScriptRunConfig
import os

# Specify parameter sampler
ps = RandomParameterSampling({
    "--C": choice(0.01, 0.1, 1.0, 10.0),
    "--max_iter": choice(50, 100, 150, 200)
}) ### YOUR CODE HERE ###

# Specify a Policy
policy = BanditPolicy(evaluation_interval=2, slack_factor=0.1) ### YOUR CODE HERE ###

if "training" not in os.listdir():
    os.mkdir("./training")

# Setup environment for your training run
sklearn_env = Environment.from_conda_specification(name='sklearn-env', file_path='conda_dependencies.yml')

# Create a ScriptRunConfig Object to specify the configuration details of your training job
src = ScriptRunConfig(
    source_directory='.',
    script='train_5June.py',
    compute_target=cpu_cluster, 
    environment=sklearn_env
) ### YOUR CODE HERE ###

# Create a HyperDriveConfig using the src object, hyperparameter sampler, and policy.
hyperdrive_config = HyperDriveConfig(
    run_config=src,
    hyperparameter_sampling=ps,
    policy=policy,
    primary_metric_name="Accuracy",
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    max_total_runs=20,
    max_concurrent_runs=4

) ### YOUR CODE HERE ###

In [17]:
# Submit your hyperdrive run to the experiment and show run details with the widget.

### YOUR CODE HERE ###
hyperdrive_run = exp.submit(hyperdrive_config)
# RunDetails(hyperdrive_run).show()

print(hyperdrive_run)


Run(Experiment: udacity-project,
Id: HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86,
Type: hyperdrive,
Status: Running)


In [18]:
hyperdrive_run.wait_for_completion(show_output=True)

RunId: HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86
Web View: https://ml.azure.com/runs/HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86?wsid=/subscriptions/d7f39349-a66b-446e-aba6-0053c2cf1c11/resourcegroups/aml-quickstarts-302116/workspaces/quick-starts-ws-302116&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254

Streaming azureml-logs/hyperdrive.txt

[2026-07-10T13:01:59.8130769Z][GENERATOR][WARNING]Space size : 16 is less than max total jobs : 20, only 16 jobs will be generated 
[2026-07-10T13:02:01.2719786Z][GENERATOR][DEBUG]Sampled 4 jobs from search space 
[2026-07-10T13:02:01.6922590Z][SCHEDULER][INFO]Scheduling job, id='HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86_0' 
[2026-07-10T13:02:01.6932750Z][SCHEDULER][INFO]Scheduling job, id='HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86_2' 
[2026-07-10T13:02:01.6941209Z][SCHEDULER][INFO]Scheduling job, id='HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86_1' 
[2026-07-10T13:02:01.7680179Z][SCHEDULER][INFO]Scheduling job, id='HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86_3' 
[2026-07-

{'runId': 'HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86',
 'target': 'Udacity-Project',
 'status': 'Completed',
 'startTimeUtc': '2026-07-10T13:01:59.067778Z',
 'endTimeUtc': '2026-07-10T13:07:06.759267Z',
 'services': {},
 'properties': {'primary_metric_config': '{"name":"Accuracy","goal":"maximize"}',
  'resume_from': 'null',
  'runTemplate': 'HyperDrive',
  'azureml.runsource': 'hyperdrive',
  'platform': 'AML',
  'ContentSnapshotId': '4c2836a8-f41b-4bfa-bc0a-d810f4b7583b',
  'user_agent': 'python/3.10.20 (Linux-6.8.0-1059-azure-x86_64-with-glibc2.35) msrest/0.7.1 Hyperdrive.Service/1.0.0 Hyperdrive.SDK/core.1.61.0',
  'best_child_run_id': 'HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86_12',
  'score': '0.9088012139605464',
  'best_metric_status': 'Succeeded',
  'best_data_container_id': 'dcid.HD_624b7c7a-29bb-4df4-9c5c-784674f0ef86_12'},
 'inputDatasets': [],
 'outputDatasets': [],
 'runDefinition': {'configuration': None,
  'attribution': None,
  'telemetryValues': {'amlClientType': 'azureml-

In [25]:
import joblib
# Get your best run and save the model from that run.

### YOUR CODE HERE ###

best_run = hyperdrive_run.get_best_run_by_primary_metric()

model = best_run.register_model(
    model_name = "hyperdrive-best-model",
    model_path = "outputs/model.joblib"
)

print(best_run.get_metrics())


{'Max iterations:': 100, 'Regularization Strength:': 0.01, 'Accuracy': 0.9088012139605463}


In [30]:
# from azureml.data.dataset_factory import TabularDatasetFactory 

## I CANNOT USE TABULARDATASETFACTORY BECAUSE OF THE KERNEL ONLY BEING AVAILABLE IN SDK2

# Create TabularDataset using TabularDatasetFactory
# Data is available at: 
# "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"

### YOUR CODE HERE ###

import pandas as pd

data_url = "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"

data = pd.read_csv(data_url)
    

In [33]:
from train_5June import clean_data

# Use the clean_data function to clean your data.
x, y = clean_data(data)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [38]:
print(azure.ai.ml.__version__)

1.34.0


In [44]:
import azureml.train.automl
from azureml.train.automl import AutoMLConfig

# Set parameters for AutoMLConfig
# NOTE: DO NOT CHANGE THE experiment_timeout_minutes PARAMETER OR YOUR INSTANCE WILL TIME OUT.
# If you wish to run the experiment longer, you will need to run this notebook in your own
# Azure tenant, which will incur personal costs.
automl_config = AutoMLConfig(
    experiment_timeout_minutes=30,
    task="classification",
    primary_metric="Accuracy",
    training_data=data,
    label_column_name="y",
    n_cross_validations=5,
    compute_target = cpu_cluster)



ModuleNotFoundError: No module named 'pkg_resources'

In [2]:
# Submit your automl run

### YOUR CODE HERE ###

In [ ]:
# Retrieve and save your best automl model.

### YOUR CODE HERE ###